# The FieldML Evaluator Graph

A Document is more than a mesh - it's a graph of named *evaluators*. This notebook walks through the evaluator subtypes, **draws the graph**, and shows how they compose.

In [ ]:
from pyfieldml import datasets

doc = datasets.load_unit_cube()

## Types vs evaluators

A Document contains:
- **Types**: BooleanType, EnsembleType, ContinuousType, MeshType.
- **Evaluators**: the graph that produces values over those types.

In [ ]:
print("Types:")
for name in doc.region.continuous:
    print("  continuous :", name)
for name in doc.region.ensembles:
    print("  ensemble   :", name)
for name in doc.region.meshes:
    print("  mesh       :", name)

## Evaluator subtypes - counts by kind

In [ ]:
from collections import Counter

kinds = Counter(type(ev).__name__ for ev in doc.region.evaluators.values())
for k, n in kinds.items():
    print(f"  {k:25s}  x{n}")

## Bar chart of evaluator kinds across every bundled dataset

This is a quick way to see *what kinds of evaluators* the model zoo actually uses - and where the complexity lives.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rows = []
for name in datasets.list():
    d = datasets.load(name)
    c = Counter(type(ev).__name__ for ev in d.region.evaluators.values())
    rows.append((name, c))

kinds_all = sorted({k for _, c in rows for k in c})
M = np.array([[r[1].get(k, 0) for k in kinds_all] for r in rows])

fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(rows))
for j, k in enumerate(kinds_all):
    ax.bar([r[0] for r in rows], M[:, j], bottom=bottom, label=k)
    bottom += M[:, j]
ax.set_ylabel("evaluator count")
ax.set_title("Evaluator composition across the bundled model zoo")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(fontsize=8)
fig.tight_layout()

## Draw the evaluator DAG

Every ParameterEvaluator in a FieldML document is wired to a data array and (for fields that live on a mesh) to a connectivity evaluator and a basis ExternalEvaluator. We expose those edges by inspecting the evaluator-name prefixes (`coordinates` depends on `coordinates.connectivity`) and by reading the region's basis list.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def build_dag(doc):
    g = nx.DiGraph()
    for name, ev in doc.region.evaluators.items():
        g.add_node(name, kind=type(ev).__name__)
    # ParameterEvaluators named 'foo' depend on 'foo.connectivity'
    # and on any basis ExternalEvaluator in the same region.
    names = set(doc.region.evaluators)
    externals = [
        n for n, ev in doc.region.evaluators.items() if type(ev).__name__ == "ExternalEvaluator"
    ]
    for name, ev in doc.region.evaluators.items():
        if type(ev).__name__ != "ParameterEvaluator":
            continue
        conn = f"{name}.connectivity"
        if conn in names:
            g.add_edge(conn, name)
            for ext in externals:
                g.add_edge(ext, name)
    return g


doc_femur = datasets.load_femur()
graph = build_dag(doc_femur)
print("nodes:", graph.number_of_nodes(), "edges:", graph.number_of_edges())

In [ ]:
kind_colors = {
    "ParameterEvaluator": "#4C78A8",
    "ExternalEvaluator": "#F58518",
    "ReferenceEvaluator": "#72B7B2",
    "AggregateEvaluator": "#E45756",
    "ArgumentEvaluator": "#54A24B",
}

fig, ax = plt.subplots(figsize=(9, 5))
try:
    pos = nx.nx_agraph.graphviz_layout(graph, prog="dot")
except (ImportError, Exception):
    pos = nx.spring_layout(graph, seed=0, k=1.2)

node_colors = [kind_colors.get(graph.nodes[n]["kind"], "lightgray") for n in graph.nodes]
nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=1400, ax=ax)
nx.draw_networkx_edges(graph, pos, arrows=True, arrowstyle="->", ax=ax)
nx.draw_networkx_labels(graph, pos, font_size=8, ax=ax)

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markerfacecolor=c,
        markeredgecolor="black",
        markersize=10,
        label=k,
    )
    for k, c in kind_colors.items()
]
ax.legend(handles=legend_handles, fontsize=8, loc="upper right")
ax.set_title("Evaluator DAG - load_femur()")
ax.axis("off")
fig.tight_layout()

## Probing a coordinate field

Even a unit cube has a full evaluator graph you can probe at any parametric location:

In [ ]:
import numpy as np

coords = doc.field("coordinates")
xi = np.array([[0.25, 0.25, 0.25], [0.5, 0.5, 0.5], [0.75, 0.75, 0.75]])
for point in xi:
    val = coords.evaluate(element=1, xi=point)
    print(f"xi={point}  ->  {val}")

## Jacobians

For a unit-cube mesh the identity map means the Jacobian is the identity matrix everywhere:

In [ ]:
J = coords.jacobian(element=1, xi=(0.5, 0.5, 0.5))
print("Jacobian at centroid:")
print(J)